[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 Medium: Implement BatchNorm

Реализуйте **Batch Normalization** с разным поведением в **training** и **inference**. Для входа `(N, D)` каждая feature-независимо нормализуется по объектам batch, то есть statistics вычисляются по `dim=0`.

In training mode, use **batch statistics** and update running estimates:

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

где $\mu_B$ и $\sigma_B^2$ — mean и variance по batch (`dim=0`). После стандартизации обучаемые `gamma` и `beta` возвращают модели возможность менять масштаб и сдвиг каждой feature.

### Почему train и eval различаются
В train доступны statistics текущего mini-batch, но на inference результат не должен зависеть от соседних объектов. Поэтому train одновременно нормализует по batch-statistics и обновляет `running_mean`/`running_var`; eval использует только накопленные running-statistics и не изменяет их.

### Momentum running statistics
$$r_{new} = (1-momentum)r_{old} + momentum\,s_{batch}$$

Это momentum для экспоненциального среднего, а не momentum optimizer. По контракту этой задачи и mean, и variance batch вычисляются с `unbiased=False`, затем именно эти значения записываются в running buffers.

### Broadcasting и autograd
`mean`, `var`, `gamma`, `beta` имеют форму `(D,)` и broadcast по размерности `N`. Gradient должен проходить к `x`, `gamma`, `beta`. Running statistics — состояние, а не обучаемые параметры; обновляйте их in-place вне gradient-графа.

### Контракт
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — normalize each feature across all samples in the batch
    # running_mean, running_var: updated in-place during training; used as-is during inference
```

### Ограничения
- не используйте `F.batch_norm`, `nn.BatchNorm1d` и другие готовые BatchNorm;
- mean и variance вычисляйте по `dim=0`, variance с `unbiased=False`;
- running statistics обновляйте по формуле выше только в train;
- в eval используйте running statistics без изменений;
- сохраняйте autograd для `x`, `gamma`, `beta`.

### Быстрые самопроверки
- при `gamma=1`, `beta=0` train-выход имеет mean около нуля по каждой feature;
- running statistics после train сдвигаются к batch-statistics;
- eval-выход не меняется при добавлении других объектов в batch;
- shape выхода совпадает с shape входа.

### Типичные ошибки
- нормализация по `dim=-1`, как в LayerNorm;
- использование batch-statistics в eval;
- обновление running buffers с gradient tracking;
- перепутанная формула momentum;
- variance без `eps`, приводящая к делению на ноль.

### Официальные материалы и примеры
- [`torch.nn.BatchNorm1d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html) — train/eval semantics, shapes и momentum;
- [`torch.nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) — параметры, buffers и состояние модуля.

> В промышленном `BatchNorm1d` есть дополнительные детали оценки running variance. Для прохождения этого упражнения следуйте явно заданному контракту judge.

### Вопросы для самопроверки
1. Почему BatchNorm связывает объекты внутри одного batch в train?
2. Почему running statistics не оптимизируются через gradient descent?
3. Чем ось нормализации BatchNorm отличается от LayerNorm?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_batch_norm(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    pass  # Replace this

In [ ]:
# 🧪 Debug
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# Running stats typically live on the same device and shape as features
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training mode: uses batch stats and updates running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out_train.shape)
print("[Train] Column means:", out_train.mean(dim=0))   # should be ~0
print("[Train] Column stds: ", out_train.std(dim=0))    # should be ~1
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference mode: uses running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Output shape:", out_eval.shape)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("batchnorm")